In [ ]:
!pip install PyPDF2 pdf2image pytesseract tqdm

!apt-get update -qq
!apt-get install -y poppler-utils tesseract-ocr tesseract-ocr-fas

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 232.6/232.6 kB 3.9 MB/s eta 0:00:00
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
tesseract-ocr is already the newest version (4.1.1-2.1build1).
The following NEW packages will be installed:
  poppler-utils tesseract-ocr-fas
0 upgraded, 2 newly installed, 0 to remove and 50 not upgraded.
Need to get 487 kB of archives.
After this operation, 1,144 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy-updates/main amd64 poppler-utils amd64 22.02.0-2ubuntu0.12 [186 kB]
Get:2 http://archive.ubuntu.com/ubuntu jammy/universe amd64 tesseract-ocr-fas all 1:4.00~git30-7274cfa-1.1 [301 kB]
Fetched 487 kB in 1s (479 kB/s)
Selecting previously unselected package poppler-utils.
(Reading 

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os, csv, re, unicodedata
from PyPDF2 import PdfReader
from pdf2image import convert_from_path
import pytesseract
from tqdm import tqdm
from multiprocessing import Pool, cpu_count
import multiprocessing

pytesseract.pytesseract.tesseract_cmd = "/usr/bin/tesseract"


def clean_persian_text(raw_text: str) -> str:
    """Clean and normalize Persian text extracted from PDFs/OCR."""
    text = re.sub(r"http\S+|www\S+|\S*@\S*", " ", raw_text)
    text = re.sub(r"[A-Za-z0-9]", " ", text)
    text = re.sub(r"[«»“”\"'؛:,،؛؛%××~_+–—\-=/\\><\[\]\{\}\(\)\|]", " ", text)
    text = re.sub(r"[•·●►©®°†‡§¤¢€£¥áó&]", " ", text)
    text = re.sub(r"[\.،]{2,}", " ", text)
    text = re.sub(r"\s[،.]\s", " ", text)
    text = re.sub(r"\s[. .]+\s", " ", text)
    text = unicodedata.normalize("NFKC", text)
    text = re.sub(r"[\u200d]+", "‌", text)      # ZWJ
    text = re.sub(r"[\u200c]{2,}", "‌", text)   # multiple ZWNJ -> single
    return re.sub(r"\s+", " ", text).strip()


def score_text_quality(text: str) -> float:
    """Heuristic score for Persian text quality."""
    if not text or len(text.strip()) < 200:
        return 0.0

    persian_chars = re.findall(r"[\u0600-\u06FF]", text)
    persian_ratio = len(persian_chars) / max(len(text), 1)

    garbage_ratio = len(re.findall(r"[A-Za-z@#_~$%&*+=<>]", text)) / max(len(text), 1)
    unreadable = len(re.findall(r"[^\x20-\x7E\u0600-\u06FF\s]", text))
    unreadable_ratio = unreadable / max(len(text), 1)

    length_score = min(len(text) / 10000, 1.0)
    repetition_penalty = len(re.findall(r"(.)\1{3,}", text)) / 50

    score = (
        persian_ratio * 0.55
        + length_score * 0.35
        - garbage_ratio * 0.4
        - repetition_penalty
        - unreadable_ratio * 0.5
    )
    return round(max(score, 0.0), 4)


def extract_text_from_pdf(pdf_path: str):
    best_text, best_score = "", 0.0
    text_pypdf2, text_ocr = "", ""
    score_pypdf2, score_ocr = 0.0, 0.0

    # --- Try PyPDF2 ---
    try:
        reader = PdfReader(pdf_path)
        text_pypdf2 = "\n".join(page.extract_text() or "" for page in reader.pages)
        text_pypdf2 = clean_persian_text(text_pypdf2)
        score_pypdf2 = score_text_quality(text_pypdf2)
        best_text, best_score = text_pypdf2, score_pypdf2
    except Exception as e:
        print(f"[PyPDF2] Failed on {pdf_path}: {e}")

    NEED_OCR = (best_score < 0.6) or (len(best_text) < 200)

    method = "PyPDF2 (no OCR)"

    if NEED_OCR:
        method = "OCR try"
        try:
            pages = convert_from_path(pdf_path, dpi=300)
            text_ocr = ""
            for page in pages:
                text_ocr += pytesseract.image_to_string(page, lang="fas+eng") + "\n"
            text_ocr = clean_persian_text(text_ocr)
            score_ocr = score_text_quality(text_ocr)

            # --- choose best ---
            if score_ocr > best_score:
                best_text, best_score = text_ocr, score_ocr
                method = "OCR (better)"
            else:
                method = "PyPDF2 (better, OCR worse)"
        except Exception as e:
            print(f"[OCR] Failed on {pdf_path}: {e}")
            method = "PyPDF2 (OCR failed)"

    return best_text, method, score_pypdf2, score_ocr, best_score



def worker(args):
    """
    Worker function for multiprocessing.
    args = (pdf_path, output_dir)
    """
    pdf_path, output_dir = args
    dirpath, filename = os.path.split(pdf_path)
    base_name, _ = os.path.splitext(filename)
    output_path = os.path.join(output_dir, f"{base_name}.txt")

    if os.path.exists(output_path):
        return [dirpath, filename, "Skipped", 0, 0, 0, output_path]

    try:
        text, method, s1, s2, s_best = extract_text_from_pdf(pdf_path)
        print("Method:", method)
        print("PyPDF2 score:", s1)
        print("OCR score:", s2)
        print("Best score:", s_best)
        with open(output_path, "w", encoding="utf-8") as f:
            f.write(text or "")
        return [dirpath, filename, method, s1, s2, s_best, output_path]
    except Exception as e:
        return [dirpath, filename, "Error", 0, 0, 0, str(e)]


def process_all_pdfs(root_folder: str):
    """
    Process all PDF files under root_folder in parallel (Colab-friendly).
    Outputs:
      - One .txt per PDF in <root_folder>/Output_Texts
      - A CSV summary in <root_folder>/pdf_extraction_summary.csv
    """
    output_dir = "/content/drive/MyDrive/Base Model Farsi/Output Texts"
    os.makedirs(output_dir, exist_ok=True)

    summary_path = os.path.join(root_folder, "pdf_extraction_summary.csv")

    pdf_files = []
    for dirpath, _, filenames in os.walk(root_folder):
        for filename in filenames:
            if filename.lower().endswith(".pdf"):
                pdf_files.append(os.path.join(dirpath, filename))

    print(f"Found {len(pdf_files)} PDF files under {root_folder}")

    WORKERS = 4
    num_workers = min(WORKERS, cpu_count())
    print(f"Using {num_workers} CPU cores\n")

    processed = set()
    if os.path.exists(summary_path):
        with open(summary_path, "r", encoding="utf-8") as f:
            next(f, None)  # skip header
            for line in f:
                parts = line.split(",")
                if len(parts) > 1:
                    processed.add(parts[1].strip())

    tasks = []
    for pdf_path in pdf_files:
        dirpath, filename = os.path.split(pdf_path)
        if filename in processed:
            continue
        tasks.append((pdf_path, output_dir))

    print(
        f"Starting extraction on {len(tasks)} new PDFs "
        f"(skipping {len(processed)} already processed)\n"
    )

    with Pool(processes=num_workers) as pool, open(
        summary_path, "a", newline="", encoding="utf-8"
    ) as csvfile:
        writer = csv.writer(csvfile)

        if os.path.getsize(summary_path) == 0:
            writer.writerow(
                [
                    "DirPath",
                    "File",
                    "Method",
                    "PyPDF2_Score",
                    "OCR_Score",
                    "Final_Score",
                    "Output_Path",
                ]
            )

        for result in tqdm(
            pool.imap_unordered(worker, tasks),
            total=len(tasks),
            desc="Extracting PDFs",
            ncols=100,
        ):
            writer.writerow(result)
            tqdm.write(f"Processed: {result[1]} ({result[2]}, score={result[5]})")

    print(f"\nAll done. Summary saved to: {summary_path}")
    print(f"Texts saved under: {output_dir}")


In [ ]:
try:
    multiprocessing.set_start_method("fork", force=True)

except RuntimeError as e:
    print("Multiprocessing start method not changed:", e)

ROOT_FOLDER = "/content/drive/MyDrive/Base Model Farsi/Documents"
process_all_pdfs(ROOT_FOLDER)

Found 1024 PDF files under /content/drive/MyDrive/Base Model Farsi/Documents
Using 2 CPU cores

Starting extraction on 1024 new PDFs (skipping 0 already processed)



Extracting PDFs:   0%|                                                     | 0/1024 [00:00<?, ?it/s]/usr/local/lib/python3.12/dist-packages/PyPDF2/_cmap.py:142: PdfReadWarning: Advanced encoding /SymbolSetEncoding not implemented yet
  warnings.warn(


Method: PyPDF2 (no OCR)
PyPDF2 score: 0.788
OCR score: 0.0
Best score: 0.788


Extracting PDFs:   0%|                                           | 1/1024 [00:12<3:27:25, 12.17s/it]

Processed: Immunization Guide line-Final-1403.pdf (PyPDF2 (no OCR), score=0.788)
Method: PyPDF2 (better, OCR worse)
PyPDF2 score: 0.5226
OCR score: 0.4857
Best score: 0.5226


...

Processed: 11. GAD7کمپ_خجک_امیر_عباس_بشارتی_فارسی_.pdf (OCR (better), score=0.4413)
Method: OCR (better)
PyPDF2 score: 0.0
OCR score: 0.7672
Best score: 0.7672


Extracting PDFs: 100%|███████████████████████████████████████▉| 1021/1024 [7:19:08<01:38, 32.74s/it]

Processed: دستورالعمل ارزشیابی رفتار حرفه ای دستیاران.pdf (OCR (better), score=0.7672)
Method: PyPDF2 (better, OCR worse)
PyPDF2 score: 0.5797
OCR score: 0.5732
Best score: 0.5797


Extracting PDFs: 100%|███████████████████████████████████████▉| 1022/1024 [7:19:21<00:53, 26.89s/it]

Processed: 12. فراکس_خطر_اوستیو_پوروز_دکتر_امیر_عباس_بشارتی_کل_250805_075349.pdf (PyPDF2 (better, OCR worse), score=0.5797)


_______
# 1404/08/28
# Post Process Text

In [ ]:
import re, os, unicodedata, csv
from tqdm import tqdm

PERSIAN_RANGE = r"\u0600-\u06FF"

ALLOWED_PUNCT = set(".,،؟!٪:%()[]-/«»\"'،-")
ALLOWED_SPACE = set([" ", "\n", "\r", "\t"])

def normalize_persian_chars(text: str) -> str:

    text = unicodedata.normalize("NFKC", text)

    text = text.replace("ي", "ی").replace("ى", "ی").replace("ئ", "ی")
    text = text.replace("ك", "ک")

    text = "".join(ch for ch in text if unicodedata.category(ch) != "Cf")

    cleaned_chars = []
    for ch in text:
        ord_ch = ord(ch)

        if re.match(f"[{PERSIAN_RANGE}]", ch):
            cleaned_chars.append(ch)
            continue

        if ch.isdigit():
            cleaned_chars.append(ch)
            continue

        if ch in ALLOWED_SPACE:
            cleaned_chars.append(ch)
            continue

        if ch in ALLOWED_PUNCT:
            cleaned_chars.append(ch)
            continue

        cleaned_chars.append(" ")

    text = "".join(cleaned_chars)

    text = re.sub(r"[ \t]+", " ", text)
    return text


def is_mostly_persian(text: str, threshold: float = 0.7) -> bool:
    if not text:
        return False
    persian_chars = re.findall(f"[{PERSIAN_RANGE}]", text)
    persian_ratio = len(persian_chars) / len(text)
    return persian_ratio >= threshold


def post_process_text(text: str) -> str:

    text = normalize_persian_chars(text)
    lines = [ln.strip() for ln in text.splitlines()]
    lines = [ln for ln in lines if ln]

    freq = {}
    for ln in lines:
        freq[ln] = freq.get(ln, 0) + 1

    cleaned_lines = []
    for ln in lines:
        if freq[ln] >= 4 and 5 <= len(ln) <= 80:
            continue

        if re.fullmatch(r"[0-9۰-۹]+", ln):
            continue

        if len(ln) < 5 and not re.search(f"[{PERSIAN_RANGE}]", ln):
            continue

        if not is_mostly_persian(ln, threshold=0.7):
            continue

        cleaned_lines.append(ln)

    merged_lines = []
    buffer = ""
    sentence_end = re.compile(r"[\.؟!:]$")

    def flush_buffer():
        nonlocal buffer
        if buffer.strip():
            merged_lines.append(buffer.strip())
        buffer = ""

    for ln in cleaned_lines:
        if not buffer:
            buffer = ln
            continue

        is_title = (len(ln) < 40) and not ln.endswith("،") and not ln.endswith("و")

        if sentence_end.search(buffer) or is_title:
            flush_buffer()
            buffer = ln
        else:
            buffer += " " + ln

    flush_buffer()

    final_paragraphs = []
    max_len = 800

    for para in merged_lines:

        para = re.sub(r"\s+([\.،؟!])", r"\1", para)
        para = re.sub(r"\s+", " ", para).strip()
        if not para:
            continue

        if len(para) > max_len:
            current = ""
            for token in re.split(r"(\s+)", para):
                if len(current) + len(token) > max_len and current:
                    final_paragraphs.append(current.strip())
                    current = token
                else:
                    current += token
            if current.strip():
                final_paragraphs.append(current.strip())
        else:
            final_paragraphs.append(para)

    return "\n\n".join(final_paragraphs)


def load_good_files_from_csv(summary_csv_path: str, min_score: float = 0.3):
    """
    Returns a set of base filenames (without path) that passed quality threshold.
    If CSV doesn't exist, returns None (meaning 'process all').
    """

    if not os.path.exists(summary_csv_path):
        print(f"Summary CSV not found at {summary_csv_path}, will process all txt files.")
        return None

    good_files = set()
    with open(summary_csv_path, "r", encoding="utf-8") as f:
        reader = csv.DictReader(f)
        for row in reader:
            try:
                score = float(row.get("Final_Score", "0") or 0)
            except ValueError:
                score = 0
            if score >= min_score:
                base = os.path.basename(row["Output_Path"])
                good_files.add(base)
    print(f"{len(good_files)} files passed Final_Score ≥ {min_score}")
    return good_files


def postprocess_pipeline(input_dir, output_dir, summary_csv_path=None, min_score=0.3):
    os.makedirs(output_dir, exist_ok=True)

    allowed_files = load_good_files_from_csv(summary_csv_path, min_score) if summary_csv_path else None
    txt_files = [f for f in os.listdir(input_dir) if f.lower().endswith(".txt")]
    print(f"Found {len(txt_files)} .txt files in {input_dir}")

    for fname in tqdm(txt_files, desc="PostProcessing", ncols=100):
        if allowed_files is not None and fname not in allowed_files:
            continue

        in_path = os.path.join(input_dir, fname)
        out_path = os.path.join(output_dir, fname)

        with open(in_path, "r", encoding="utf-8") as f:
            raw_text = f.read()

        cleaned_text = post_process_text(raw_text)

        with open(out_path, "w", encoding="utf-8") as f:
            f.write(cleaned_text)

    print(f"\nDone. Cleaned texts saved in: {output_dir}")

In [3]:
if __name__ == "__main__":
    INPUT_DIR = "/content/drive/MyDrive/Base Model Farsi/Output Texts"
    OUTPUT_DIR = "/content/drive/MyDrive/Base Model Farsi/Output Texts Cleaned"
    summary_csv_path = "/content/drive/MyDrive/Base Model Farsi/Documents/pdf_extraction_summary.csv"

    postprocess_pipeline(INPUT_DIR, OUTPUT_DIR, summary_csv_path)

885 files passed Final_Score ≥ 0.3
Found 886 .txt files in /content/drive/MyDrive/Base Model Farsi/Output Texts


PostProcessing: 100%|█████████████████████████████████████████████| 886/886 [07:15<00:00,  2.03it/s]


Done. Cleaned texts saved in: /content/drive/MyDrive/Base Model Farsi/Output Texts Cleaned
